# Day 13: Speculative Decoding — Draft-Target, Medusa, EAGLE
> *Inference Engineering* — Chapter 5.2 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 11 (Quantization), Day 01 (Decoding)


## Concept Overview

Speculative decoding accelerates autoregressive generation without changing output distribution. A small draft model proposes K tokens; the large target model verifies all K+1 positions in one parallel forward pass. If all K tokens are accepted, we get K+1 tokens in the time of 2 model calls (vs K+1 calls normally). Medusa adds multiple prediction heads to the base model itself, eliminating the separate draft model. EAGLE improves draft quality by conditioning on the target model's hidden states rather than just output tokens.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Speculative Decoding Algorithm

The acceptance sampling criterion ensures the output distribution matches the target model exactly:

$$p_{\text{accept}}(x) = \min\left(1, \frac{p_{\text{target}}(x)}{p_{\text{draft}}(x)}\right)$$

Rejected tokens are resampled from a corrected distribution, maintaining exact target distribution matching.


In [ ]:
def speculative_decode_step(target_logits, draft_logits, draft_token, temperature=1.0):
    """
    One step of speculative decoding acceptance sampling.
    Returns (accepted, sampled_token)
    """
    p_target = F.softmax(target_logits / temperature, dim=-1)
    p_draft = F.softmax(draft_logits / temperature, dim=-1)

    # Acceptance probability
    accept_prob = torch.min(torch.ones_like(p_target), p_target / (p_draft + 1e-8))
    p_accept = accept_prob[draft_token].item()

    # Accept with probability min(1, p_target/p_draft)
    if torch.rand(1).item() < p_accept:
        return True, draft_token
    else:
        # Resample from corrected distribution
        p_corrected = torch.clamp(p_target - p_draft, min=0)
        p_corrected = p_corrected / p_corrected.sum()
        resampled = torch.multinomial(p_corrected, 1).item()
        return False, resampled

# Simulate speculative decoding over 100 steps
torch.manual_seed(42)
vocab_size = 32000
K = 4  # draft tokens per speculation step

def run_speculation(num_steps=200, K=4, draft_quality=0.7):
    """
    Simulate speculative decoding.
    draft_quality: probability that draft token matches target's top-1
    """
    total_tokens = 0
    total_steps = 0
    accept_counts = []

    for _ in range(num_steps):
        # Draft proposes K tokens
        # Simulate: each token accepted with prob related to draft quality
        n_accepted = 0
        for k in range(K):
            target_logits = torch.randn(vocab_size)
            draft_logits = target_logits.clone()
            # Reduce draft quality by adding noise
            draft_logits += torch.randn(vocab_size) * (1 - draft_quality) * 3
            draft_tok = torch.argmax(F.softmax(draft_logits, dim=-1)).item()
            accepted, _ = speculative_decode_step(target_logits, draft_logits, draft_tok)
            if accepted:
                n_accepted += 1
            else:
                break  # rejection stops the chain

        tokens_this_step = n_accepted + 1  # +1 for the always-generated bonus token
        total_tokens += tokens_this_step
        total_steps += 1
        accept_counts.append(n_accepted)

    return total_tokens / total_steps, np.mean(accept_counts)

print('Speculative decoding simulation:')
print(f'{"Draft Quality":>15} {"Tokens/Step":>12} {"Mean Accepts":>13} {"Speedup":>10}')
for quality in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    tps, acc = run_speculation(draft_quality=quality)
    print(f'{quality:>15.2f} {tps:>12.2f} {acc:>13.2f} {tps:>10.2f}x')


## 2. Acceptance Rate vs Speedup

The expected speedup with acceptance rate $\alpha$ and K draft tokens is:
$$\text{Speedup} = \frac{1 - \alpha^{K+1}}{(1-\alpha)(1 + c)}$$
where $c$ is the cost ratio (draft model time / target model time).


In [ ]:
def expected_speedup(alpha, K, cost_ratio=0.1):
    """
    Expected tokens per second improvement.
    alpha: per-token acceptance rate
    K: number of draft tokens
    cost_ratio: draft_model_time / target_model_time
    """
    # Expected accepted tokens per speculation step
    expected_accepted = sum((alpha**k) * (1 - (alpha if k < K else 0)) * k
                           for k in range(K+1)) + alpha**K * K
    # Simplified: E[accepted] ≈ (1 - alpha^(K+1)) / (1 - alpha)
    e_accepted = (1 - alpha**(K+1)) / (1 - alpha + 1e-8)
    tokens_per_step = e_accepted + 1  # +1 bonus
    step_cost = K * cost_ratio + 1  # K draft calls + 1 target call
    return tokens_per_step / step_cost

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

alphas = np.linspace(0.5, 0.99, 100)
for K in [1, 2, 4, 8]:
    speedups = [expected_speedup(a, K) for a in alphas]
    ax1.plot(alphas, speedups, label=f'K={K}')
ax1.set_xlabel('Acceptance Rate α')
ax1.set_ylabel('Expected Speedup')
ax1.set_title('Speculative Decoding Speedup\n(cost_ratio=0.1)')
ax1.legend()
ax1.grid(True)
ax1.axhline(1.0, color='black', linestyle='--', alpha=0.5)

Ks = range(1, 17)
for alpha in [0.7, 0.8, 0.9]:
    speedups = [expected_speedup(alpha, K) for K in Ks]
    ax2.plot(Ks, speedups, label=f'α={alpha}')
ax2.set_xlabel('Draft Tokens K')
ax2.set_ylabel('Expected Speedup')
ax2.set_title('Optimal K vs Acceptance Rate')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('speculative_decoding.png', dpi=100, bbox_inches='tight')
plt.show()


## 3. Medusa: Multi-Head Speculation

Medusa adds K independent prediction heads on top of the base model, each predicting the token at position t+k. No separate model — the heads share the base model's hidden states. Tree-based verification allows parallel validation of multiple candidate sequences.


In [ ]:
class MedusaHeads(torch.nn.Module):
    def __init__(self, hidden_size, vocab_size, num_heads=4):
        super().__init__()
        self.heads = torch.nn.ModuleList([
            torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_size),
                torch.nn.SiLU(),
                torch.nn.Linear(hidden_size, vocab_size)
            ) for _ in range(num_heads)
        ])

    def forward(self, hidden_state):
        # hidden_state: [batch, seq, hidden]
        return [head(hidden_state) for head in self.heads]

medusa = MedusaHeads(hidden_size=512, vocab_size=32000, num_heads=4)
hidden = torch.randn(1, 10, 512)
head_logits = medusa(hidden)
print(f'Medusa heads output: {len(head_logits)} heads, each {head_logits[0].shape}')
print(f'Head 0 (t+1) top-5 tokens: {head_logits[0][0,-1].topk(5).indices.tolist()}')
print(f'Head 1 (t+2) top-5 tokens: {head_logits[1][0,-1].topk(5).indices.tolist()}')

params_base = 512 * 32000 * 2  # approximate base LM head
params_medusa = sum(p.numel() for p in medusa.parameters())
print(f'\nMedusa head overhead: {params_medusa/params_base:.1f}x base LM head size')
print(f'In practice: ~1-2% of total model parameters')


## Experiments: Try These

1. **Acceptance rate by domain**: Sample text from code vs prose vs math. Compare simulated acceptance rates — where does speculative decoding help most?
2. **EAGLE draft**: EAGLE uses the target model's hidden states as context for the draft model. Why does this improve acceptance rate vs token-only drafting?
3. **Tree speculation**: Implement tree-based speculation where K heads generate a tree of candidates. How many tokens can you verify with one target forward pass?


## Key Takeaways

- Speculative decoding verifies K draft tokens in a single target model forward pass — the output distribution is mathematically identical to standard decoding.
- Expected speedup = (1 + mean_accepted_tokens) / (1 + K * cost_ratio) — acceptance rate and draft cost both matter.
- Medusa eliminates the draft model by adding prediction heads to the base model itself, trading head training cost for zero draft model memory.
- EAGLE and similar methods improve acceptance rate by conditioning the draft on target model hidden states, not just output tokens.

## References
- *Inference Engineering* Ch 5.2 — Philip Kiely, Baseten Books 2026
- Leviathan et al. (2022), "Fast Inference from Transformers via Speculative Decoding"
- Cai et al. (2023), "Medusa: Simple LLM Inference Acceleration Framework"
- Li et al. (2024), "EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty"
